# Eval: check observed behavior · 2.1.5

Input value=3 → Agent adds one → Eval requires 4. A negative control requires 999 and must fail. Eval explicitly targets an Agent, System or Environment.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.6'


In [3]:
class State(pydantic.BaseModel):
    value: int

@toolkit.tool
def increment(value: int) -> dict:
    return {'value': value + 1}
graph = toolkit.Graph(State)
graph.add_node('increment', increment)
graph.add_edge(toolkit.START, 'increment')
graph.add_edge('increment', toolkit.END)
agent = toolkit.Agent('counter', graph=graph)


In [4]:
evaluation = toolkit.Eval('correctness', agent, cases=({'input': {'value': 3}, 'expected': {'data': {'value': 4}}},))
passed = evaluation.run()
negative = toolkit.Eval('negative', agent, cases=({'input': {'value': 3}, 'expected': {'data': {'value': 999}}},)).run()
assert passed.ok and (not negative.ok)
print({'positive': passed.ok, 'negative_detected': not negative.ok})


{'positive': True, 'negative_detected': True}


Eval stops at the first failed case. Run independent cases to measure all negative decisions. A fixture judge tests the protocol; it does not certify semantic judgment.